# Chapter 13 Lab: Language: Context and Grounding (`ch08`)

In this lab, we explore how text is tokenized, represented as vectors, and used for similarity and classification. We'll see why word order matters and how TF-IDF reveals interpretable meaning.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import re
import warnings
warnings.filterwarnings('ignore')

# Seed for reproducibility
np.random.seed(42)

## 3. Tokenization

Take a few sentences and tokenize them using whitespace and regex. Observe how tokenization choices affect vocabulary size.

In [ ]:
# Example sentences
sentences = [
    "The cat sat on the mat.",
    "The dog ran in the park.",
    "I love to eat pizza!",
    "Machine learning is fascinating.",
    "Natural language processing is hard."
]

print("=== Original Sentences ===")
for i, sent in enumerate(sentences, 1):
    print(f"{i}. {sent}")

# Tokenization approach 1: Whitespace split
print("\n=== Tokenization: Whitespace Split ===")
tokens_whitespace = [sent.split() for sent in sentences]
vocab_whitespace = set()
for tokens in tokens_whitespace:
    vocab_whitespace.update(tokens)

print(f"Vocabulary size: {len(vocab_whitespace)}")
for i, tokens in enumerate(tokens_whitespace, 1):
    print(f"{i}. {tokens} ({len(tokens)} tokens)")

# Tokenization approach 2: Regex (split punctuation)
print("\n=== Tokenization: Regex (Split Punctuation) ===")
def tokenize_regex(text):
    # Convert to lowercase, split on non-alphanumeric
    text = text.lower()
    tokens = re.findall(r'\b\w+\b', text)
    return tokens

tokens_regex = [tokenize_regex(sent) for sent in sentences]
vocab_regex = set()
for tokens in tokens_regex:
    vocab_regex.update(tokens)

print(f"Vocabulary size: {len(vocab_regex)}")
for i, tokens in enumerate(tokens_regex, 1):
    print(f"{i}. {tokens} ({len(tokens)} tokens)")

# Comparison
fig, ax = plt.subplots(figsize=(10, 5))
methods = ['Whitespace Split', 'Regex (lowercase,\nno punctuation)']
vocab_sizes = [len(vocab_whitespace), len(vocab_regex)]
colors = ['skyblue', 'lightcoral']
bars = ax.bar(methods, vocab_sizes, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Vocabulary Size', fontsize=12, fontweight='bold')
ax.set_title('Impact of Tokenization on Vocabulary', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

for bar, size in zip(bars, vocab_sizes):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(size)}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('tokenization_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n** Insight: Tokenization choices affect vocabulary and subsequent representations. **")
print(f"Regex approach reduces vocabulary by normalizing case and removing punctuation.")

## 4. Bag-of-Words Representation

Build bag-of-words vectors and compute cosine similarity between sentences.

In [ ]:
# Use sklearn's CountVectorizer for BoW
vectorizer = CountVectorizer(lowercase=True, stop_words=None)
X_bow = vectorizer.fit_transform(sentences).toarray()

vocab = vectorizer.get_feature_names_out()

print("=== Bag-of-Words Matrix ===")
print(f"Shape: {X_bow.shape} (sentences × vocabulary)")
print(f"\nVocabulary: {list(vocab)}")
print(f"\nBoW vectors:")
for i, sent in enumerate(sentences):
    print(f"{i+1}. {sent[:40]}...")
    print(f"   {X_bow[i]}\n")

# Compute pairwise cosine similarity
similarity_matrix = cosine_similarity(X_bow)

print("=== Cosine Similarity Matrix ===")
print("(Ranges from 0=orthogonal to 1=identical)\n")
print("\t", end="")
for j in range(len(sentences)):
    print(f"S{j+1}\t", end="")
print()
for i in range(len(sentences)):
    print(f"S{i+1}\t", end="")
    for j in range(len(sentences)):
        print(f"{similarity_matrix[i, j]:.3f}\t", end="")
    print()

# Visualize similarity matrix
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(similarity_matrix, cmap='RdYlGn', vmin=0, vmax=1)

# Labels
labels = [f"S{i+1}" for i in range(len(sentences))]
ax.set_xticks(range(len(sentences)))
ax.set_yticks(range(len(sentences)))
ax.set_xticklabels(labels, fontsize=11)
ax.set_yticklabels(labels, fontsize=11)
ax.set_title('Cosine Similarity: BoW Vectors', fontsize=13, fontweight='bold')

# Add text annotations
for i in range(len(sentences)):
    for j in range(len(sentences)):
        text = ax.text(j, i, f"{similarity_matrix[i, j]:.2f}",
                       ha="center", va="center", color="black", fontsize=10, fontweight='bold')

plt.colorbar(im, ax=ax, label='Similarity')
plt.tight_layout()
plt.savefig('bow_similarity.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\n** Insight: Sentences with shared words have higher similarity. **")
print(f"Sentences 1 & 2 (both about animals) are more similar than 1 & 3 (different topics).")

## 5. Word Order Matters

Show two sentences with identical words but opposite meanings. BoW treats them the same; bigrams distinguish them.

In [ ]:
# Two sentences with same words, different meanings
order_sentences = [
    "The dog bit the man.",
    "The man bit the dog."
]

print("=== Order Matters ===")
print(f"Sentence 1: {order_sentences[0]}")
print(f"Sentence 2: {order_sentences[1]}")
print(f"\nMeanings are opposite, but they share the same words!\n")

# BoW (unigrams only)
vec_unigram = CountVectorizer(ngram_range=(1, 1), lowercase=True)
X_unigram = vec_unigram.fit_transform(order_sentences).toarray()

print("=== Unigram (1-gram) Vectors ===")
print(f"Sentence 1: {X_unigram[0]}")
print(f"Sentence 2: {X_unigram[1]}")
sim_unigram = cosine_similarity(X_unigram)[0, 1]
print(f"Cosine similarity: {sim_unigram:.4f}")
print(f"→ Identical! BoW ignores word order.\n")

# Bigrams (pairs of consecutive words)
vec_bigram = CountVectorizer(ngram_range=(1, 2), lowercase=True)
X_bigram = vec_bigram.fit_transform(order_sentences).toarray()

print("=== Bigram (1-gram + 2-gram) Vectors ===")
vocab_bigram = vec_bigram.get_feature_names_out()
print(f"Vocabulary: {list(vocab_bigram)}\n")
print(f"Sentence 1: {X_bigram[0]}")
print(f"Sentence 2: {X_bigram[1]}")
sim_bigram = cosine_similarity(X_bigram)[0, 1]
print(f"Cosine similarity: {sim_bigram:.4f}")
print(f"→ Different! Bigrams capture word order.\n")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Unigram similarity
sim_matrix_uni = cosine_similarity(X_unigram)
im0 = axes[0].imshow(sim_matrix_uni, cmap='RdYlGn', vmin=0, vmax=1)
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['S1', 'S2'], fontsize=11)
axes[0].set_yticklabels(['S1', 'S2'], fontsize=11)
axes[0].set_title('Unigrams Only\n(Order ignored)', fontsize=12, fontweight='bold')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, f"{sim_matrix_uni[i, j]:.3f}",
                     ha="center", va="center", color="black", fontsize=11, fontweight='bold')
plt.colorbar(im0, ax=axes[0], label='Similarity')

# Bigram similarity
sim_matrix_bi = cosine_similarity(X_bigram)
im1 = axes[1].imshow(sim_matrix_bi, cmap='RdYlGn', vmin=0, vmax=1)
axes[1].set_xticks([0, 1])
axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(['S1', 'S2'], fontsize=11)
axes[1].set_yticklabels(['S1', 'S2'], fontsize=11)
axes[1].set_title('Unigrams + Bigrams\n(Order matters)', fontsize=12, fontweight='bold')
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f"{sim_matrix_bi[i, j]:.3f}",
                     ha="center", va="center", color="black", fontsize=11, fontweight='bold')
plt.colorbar(im1, ax=axes[1], label='Similarity')

plt.tight_layout()
plt.savefig('word_order_matters.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"** Insight: Word order is crucial for meaning. **")
print(f"Unigrams lose this information. N-grams (bigrams, trigrams) capture it.")

## 6. TF-IDF vs Raw Counts

Compare raw word counts with TF-IDF, showing how TF-IDF downweights common words and highlights distinctive terms.

In [ ]:
# Create a small document collection with some common words
documents = [
    "The cat is on the mat.",
    "The dog is in the house.",
    "Cats and dogs are animals.",
    "I love machine learning and artificial intelligence.",
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks.",
    "Neural networks learn from data.",
    "Data science involves statistics and machine learning.",
    "The weather is sunny today.",
    "Today is a beautiful day."
]

print("=== Document Collection ===")
for i, doc in enumerate(documents, 1):
    print(f"{i}. {doc}")

# Raw counts (BoW)
vec_count = CountVectorizer(lowercase=True, stop_words='english')
X_count = vec_count.fit_transform(documents).toarray()

# TF-IDF
vec_tfidf = TfidfVectorizer(lowercase=True, stop_words='english')
X_tfidf = vec_tfidf.fit_transform(documents).toarray()

vocab = vec_count.get_feature_names_out()

# Select document 4 (about machine learning) for comparison
doc_idx = 3

print(f"\n=== Document {doc_idx+1}: '{documents[doc_idx]}' ===")

# Get top features by raw count
count_features = X_count[doc_idx]
top_count_idx = np.argsort(count_features)[-5:][::-1]
print(f"\nTop 5 features (Raw Count):")
for idx in top_count_idx:
    if count_features[idx] > 0:
        print(f"  {vocab[idx]:20s}: {count_features[idx]:.1f}")

# Get top features by TF-IDF
tfidf_features = X_tfidf[doc_idx]
top_tfidf_idx = np.argsort(tfidf_features)[-5:][::-1]
print(f"\nTop 5 features (TF-IDF):")
for idx in top_tfidf_idx:
    if tfidf_features[idx] > 0:
        print(f"  {vocab[idx]:20s}: {tfidf_features[idx]:.4f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Raw counts
count_vals = count_features[top_count_idx]
count_labels = vocab[top_count_idx]
axes[0].barh(range(len(count_labels)), count_vals, color='skyblue', edgecolor='black')
axes[0].set_yticks(range(len(count_labels)))
axes[0].set_yticklabels(count_labels, fontsize=11)
axes[0].set_xlabel('Count', fontsize=11, fontweight='bold')
axes[0].set_title('Raw Counts (BoW)', fontsize=12, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)
for i, v in enumerate(count_vals):
    axes[0].text(v, i, f' {v:.0f}', va='center', fontsize=10, fontweight='bold')

# TF-IDF
tfidf_vals = tfidf_features[top_tfidf_idx]
tfidf_labels = vocab[top_tfidf_idx]
axes[1].barh(range(len(tfidf_labels)), tfidf_vals, color='lightcoral', edgecolor='black')
axes[1].set_yticks(range(len(tfidf_labels)))
axes[1].set_yticklabels(tfidf_labels, fontsize=11)
axes[1].set_xlabel('TF-IDF Score', fontsize=11, fontweight='bold')
axes[1].set_title('TF-IDF (Downweights Common Words)', fontsize=12, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)
for i, v in enumerate(tfidf_vals):
    axes[1].text(v, i, f' {v:.3f}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('tfidf_vs_counts.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\n** Insight: TF-IDF reveals distinctive terms. **")
print(f"Common words like 'is', 'the' get downweighted.")
print(f"Domain-specific words like 'learning', 'artificial' get higher scores.")

## 7. Simple Text Classifier with Interpretability

Generate sentiment snippets, train a logistic regression classifier, and inspect the most predictive words.

In [ ]:
# Generate synthetic sentiment data
np.random.seed(42)

positive_words = ['love', 'great', 'wonderful', 'excellent', 'fantastic', 'amazing', 'brilliant', 'happy', 'beautiful']
negative_words = ['hate', 'terrible', 'awful', 'horrible', 'bad', 'dreadful', 'disappointing', 'sad', 'ugly']
neutral_words = ['the', 'is', 'a', 'and', 'to', 'that', 'this', 'in', 'with']

def generate_sentiment_texts(n_samples=50, seed=42):
    np.random.seed(seed)
    texts_pos, texts_neg = [], []
    
    for _ in range(n_samples):
        # Positive texts
        pos_words_sample = np.random.choice(positive_words, size=2, replace=False)
        neutral_sample = np.random.choice(neutral_words, size=1)
        pos_text = f"{pos_words_sample[0]} movie. {neutral_sample[0]} {pos_words_sample[1]} experience!"
        texts_pos.append(pos_text)
        
        # Negative texts
        neg_words_sample = np.random.choice(negative_words, size=2, replace=False)
        neutral_sample = np.random.choice(neutral_words, size=1)
        neg_text = f"{neg_words_sample[0]} movie. {neutral_sample[0]} {neg_words_sample[1]} experience!"
        texts_neg.append(neg_text)
    
    return texts_pos + texts_neg, np.array([1]*n_samples + [0]*n_samples)

X_texts, y_sentiment = generate_sentiment_texts(n_samples=50, seed=42)

# Shuffle
indices = np.random.permutation(len(X_texts))
X_texts = [X_texts[i] for i in indices]
y_sentiment = y_sentiment[indices]

print(f"=== Generated Sentiment Dataset ===")
print(f"Total samples: {len(X_texts)}")
print(f"Positive samples: {np.sum(y_sentiment)}")
print(f"Negative samples: {len(y_sentiment) - np.sum(y_sentiment)}")
print(f"\nExamples:")
for i in range(3):
    print(f"  {X_texts[i]} → {'Positive' if y_sentiment[i] else 'Negative'}")

# Train/test split (80/20)
n_train = int(0.8 * len(X_texts))
X_train_sent, X_test_sent = X_texts[:n_train], X_texts[n_train:]
y_train_sent, y_test_sent = y_sentiment[:n_train], y_sentiment[n_train:]

# Vectorize with TF-IDF
vec = TfidfVectorizer(lowercase=True, stop_words='english', max_features=50)
X_train_tfidf = vec.fit_transform(X_train_sent).toarray()
X_test_tfidf = vec.transform(X_test_sent).toarray()

# Train logistic regression
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_tfidf, y_train_sent)

# Evaluate
train_acc = accuracy_score(y_train_sent, clf.predict(X_train_tfidf))
test_acc = accuracy_score(y_test_sent, clf.predict(X_test_tfidf))

print(f"\n=== Classifier Performance ===")
print(f"Train accuracy: {train_acc:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

# Get feature names and coefficients
feature_names = vec.get_feature_names_out()
coef = clf.coef_[0]

# Top positive and negative words
top_pos_idx = np.argsort(coef)[-5:]
top_neg_idx = np.argsort(coef)[:5]

print(f"\n=== Interpretability: Most Predictive Words ===")
print(f"\nTop 5 Positive Indicators (push towards 'Positive'):")
for idx in top_pos_idx[::-1]:
    print(f"  {feature_names[idx]:15s}: {coef[idx]:+.4f}")

print(f"\nTop 5 Negative Indicators (push towards 'Negative'):")
for idx in top_neg_idx:
    print(f"  {feature_names[idx]:15s}: {coef[idx]:+.4f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# Top positive words
pos_words = feature_names[top_pos_idx[::-1]]
pos_coef = coef[top_pos_idx[::-1]]
axes[0].barh(range(len(pos_words)), pos_coef, color='green', alpha=0.7, edgecolor='black')
axes[0].set_yticks(range(len(pos_words)))
axes[0].set_yticklabels(pos_words, fontsize=11)
axes[0].set_xlabel('Coefficient', fontsize=11, fontweight='bold')
axes[0].set_title('Most Positive Indicators', fontsize=12, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)
for i, v in enumerate(pos_coef):
    axes[0].text(v, i, f' {v:.3f}', va='center', fontsize=10, fontweight='bold')

# Top negative words
neg_words = feature_names[top_neg_idx]
neg_coef = coef[top_neg_idx]
axes[1].barh(range(len(neg_words)), neg_coef, color='red', alpha=0.7, edgecolor='black')
axes[1].set_yticks(range(len(neg_words)))
axes[1].set_yticklabels(neg_words, fontsize=11)
axes[1].set_xlabel('Coefficient', fontsize=11, fontweight='bold')
axes[1].set_title('Most Negative Indicators', fontsize=12, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)
for i, v in enumerate(neg_coef):
    axes[1].text(v, i, f' {v:.3f}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('sentiment_classifier.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\n** Insight: Linear classifiers are interpretable. **")
print(f"We can directly inspect which words drive predictions.")
print(f"This is crucial for trust and debugging in real applications!")

## What to Notice

1. **Tokenization shapes representation**: Whitespace splitting preserves punctuation as part of tokens; regex normalization removes it and converts to lowercase. The choice affects vocabulary size and downstream modeling.

2. **Bag-of-Words is order-agnostic**: BoW vectors count word occurrences but discard order. Sentences with identical words get identical vectors, even if their meanings differ.

3. **N-grams capture context**: Bigrams and higher-order n-grams preserve sequential information. "dog bit man" and "man bit dog" have different bigram features and can be distinguished.

4. **TF-IDF vs raw counts**: Raw counts favor common words; TF-IDF downweights them and highlights distinctive, document-specific terms. TF-IDF is more informative for classification and similarity tasks.

5. **Interpretability in text classifiers**: Linear models (logistic regression) trained on sparse, interpretable features (BoW, TF-IDF) allow us to inspect which words are most predictive. This is invaluable for debugging and building trust.

**Key Takeaway**: Language models begin with tokenization and vectorization. Simple representations like BoW and TF-IDF are interpretable, efficient, and often surprisingly effective. Higher-order models (like neural networks) sacrifice interpretability for capacity but require more data and computation. Understanding the tradeoffs is essential for effective NLP system design.